In [1]:
import tensorflow as tf
import gc
from tensorflow.keras import backend as K
import os

os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"

# Evitar que TensorFlow reserve toda la GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

print("GPU lista")
print(tf.config.list_physical_devices("GPU"))
def reset_tf():
    K.clear_session()
    gc.collect()

GPU lista
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
from os import listdir
from numpy import asarray
from numpy import save
import tensorflow as tf
import pandas as pd
import numpy as np
#Deshabilitar la GPU:
#tf.config.set_visible_devices([], 'GPU')
from tensorflow.keras.utils import load_img
from tensorflow.keras.utils import img_to_array

folders = listdir('./famosos/')
#Clase Kirmizi será la clase 0.0
#Clase Siirt será la clase 1.0


photos =  []
labels = []

In [3]:
#2. IMPORTAMOS LOS DATOS:
rasgos = pd.read_csv("list_attr_celeba.csv", sep=",")
rasgos = rasgos.head(70000)

photos = []

for file in rasgos["image_id"]:
    #Cargamos la imagen.
    photo = load_img('./famosos/famosos/' + file, target_size=(64, 64))
    #Convertimos la imagen a un array.
    photo = img_to_array(photo)
    #Los guardamos en la lista.
    photos.append(photo)
    del photo

In [4]:
photos = asarray(photos)
rasgos.replace(-1, 0, inplace=True)
rasgos.drop(columns=["image_id"], inplace=True, axis=1)
#rasgos = rasgos[["Attractive", "Bags_Under_Eyes", "Bald"]]
#rasgos = rasgos[["Attractive", "High_Cheekbones", "Mouth_Slightly_Open"]]

In [5]:
from sklearn.model_selection import train_test_split

# Datos originales
X = photos / 255.0
y = rasgos

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
print(X_train.shape, y_train.shape)

(56000, 64, 64, 3) (56000, 40)


In [8]:
# Haciendo la red neuronal a partir del tratamiento de PCA
from tensorflow import keras

model = keras.Sequential()
model.add(keras.layers.Flatten(input_shape=(64, 64, 3)))
# Luego metemos capas ocultas
model.add(keras.layers.Dense(512, activation="relu"))
model.add(keras.layers.Dense(256, activation="relu"))
# Luego metemos la capa de salida, que tiene 5 neuronas, una por cada clase, y función de activación softmax, que es la que se suele usar para clasificación multiclase.
model.add(keras.layers.Dense(40, activation="sigmoid"))

from tensorflow.keras import optimizers
sgd = optimizers.Adam(learning_rate=0.00005)

model.compile(loss="binary_crossentropy", optimizer=sgd, metrics=[keras.metrics.BinaryAccuracy(name="accuracy")])

In [9]:
early_stopping_cb = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=30, validation_split=0.1, callbacks=[early_stopping_cb])

Epoch 1/30


2026-04-23 18:59:14.292066: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2026-04-23 18:59:14.783775: I external/local_xla/xla/service/service.cc:168] XLA service 0x7741ba3dbea0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-04-23 18:59:14.783798: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 3060, Compute Capability 8.6
2026-04-23 18:59:14.789910: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-04-23 18:59:14.800629: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8904
I0000 00:00:1776963554.861232  129346 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1575/1575 [==============================] - 6s 3ms/step - loss: 0.3621 - accuracy: 0.8467 - val_loss: 0.3328 - val_accuracy: 0.8579
Epoch 2/30
1575/1575 [==============================] - 5s 3ms/step - loss: 0.3162 - accuracy: 0.8648 - val_loss: 0.3089 - val_accuracy: 0.8684
Epoch 3/30
1575/1575 [==============================] - 4s 3ms/step - loss: 0.3014 - accuracy: 0.8703 - val_loss: 0.2987 - val_accuracy: 0.8706
Epoch 4/30
1575/1575 [==============================] - 4s 3ms/step - loss: 0.2931 - accuracy: 0.8733 - val_loss: 0.2927 - val_accuracy: 0.8731
Epoch 5/30
1575/1575 [==============================] - 5s 3ms/step - loss: 0.2874 - accuracy: 0.8757 - val_loss: 0.2876 - val_accuracy: 0.8750
Epoch 6/30
1575/1575 [==============================] - 4s 3ms/step - loss: 0.2827 - accuracy: 0.8777 - val_loss: 0.2865 - val_accuracy: 0.8747
Epoch 7/30
1575/1575 [==============================] - 4s 3ms/step - loss: 0.2789 - accuracy: 0.8792 - val_loss: 0.2834 - val_accuracy: 0.8774
Epo

In [10]:
model.evaluate(X_test, y_test)

438/438 [==============================] - 1s 1ms/step - loss: 0.2707 - accuracy: 0.8823


[0.27065688371658325, 0.8823214173316956]

In [11]:
del model
reset_tf()

## Vamos a hacer la red neuronal convolucional

In [12]:
from tensorflow import keras
from tensorflow.keras import layers, optimizers, initializers

# Definición de la red convolucional
model_cnn = keras.Sequential()

# Capa de convolución: el input_shape debe ser (128, 128, 1) (1 en blanco y negro, 3 en rgb)
model_cnn.add(keras.layers.Conv2D(32, (3, 3), activation="relu", input_shape=(64, 64, 3)))
model_cnn.add(keras.layers.MaxPooling2D((2, 2)))

model_cnn.add(keras.layers.Flatten())
model_cnn.add(keras.layers.Dense(768, activation="relu"))
model_cnn.add(keras.layers.Dense(512, activation="relu"))
model_cnn.add(keras.layers.Dense(40, activation="sigmoid"))

# Optimizador y compilación
sgd_cnn = optimizers.Adam(learning_rate=0.0001)
model_cnn.compile(optimizer=sgd_cnn, loss="binary_crossentropy", metrics=[keras.metrics.BinaryAccuracy(name="accuracy")])

In [13]:
early_stopping_cb = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
history = model_cnn.fit(X_train, y_train, epochs=30, validation_split=0.1, callbacks=[early_stopping_cb])

Epoch 1/30


2026-04-23 19:01:31.969956: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory


1575/1575 [==============================] - 15s 7ms/step - loss: 0.3194 - accuracy: 0.8630 - val_loss: 0.2805 - val_accuracy: 0.8789
Epoch 2/30
1575/1575 [==============================] - 11s 7ms/step - loss: 0.2667 - accuracy: 0.8840 - val_loss: 0.2639 - val_accuracy: 0.8843
Epoch 3/30
1575/1575 [==============================] - 11s 7ms/step - loss: 0.2504 - accuracy: 0.8908 - val_loss: 0.2530 - val_accuracy: 0.8885
Epoch 4/30
1575/1575 [==============================] - 11s 7ms/step - loss: 0.2392 - accuracy: 0.8957 - val_loss: 0.2492 - val_accuracy: 0.8905
Epoch 5/30
1575/1575 [==============================] - 11s 7ms/step - loss: 0.2299 - accuracy: 0.8996 - val_loss: 0.2452 - val_accuracy: 0.8917
Epoch 6/30
1575/1575 [==============================] - 11s 7ms/step - loss: 0.2214 - accuracy: 0.9033 - val_loss: 0.2433 - val_accuracy: 0.8931
Epoch 7/30
1575/1575 [==============================] - 11s 7ms/step - loss: 0.2134 - accuracy: 0.9070 - val_loss: 0.2402 - val_accuracy: 0.8

In [14]:
model_cnn.evaluate(X_test, y_test)

438/438 [==============================] - 1s 2ms/step - loss: 0.2405 - accuracy: 0.8941


[0.2404605895280838, 0.8940553665161133]

In [15]:
del model_cnn
reset_tf()